In [1]:
# Install packages
!pip install -q transformers accelerate peft pillow

# Load model with LoRA
from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from PIL import Image
import torch
import requests
from io import BytesIO

model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma-3b-pt-224",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

processor = PaliGemmaProcessor.from_pretrained("google/paligemma-3b-pt-224")

# Load images
working_data = []
test_urls = [
    ("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg", "a red car"),
    ("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beignets-task-guide.png", "beignets on a plate"),
    ("https://huggingface.co/datasets/Narsil/image_dummy/resolve/main/parrots.png", "two parrots"),
    ("https://huggingface.co/datasets/Narsil/image_dummy/resolve/main/tree.png", "a tree"),
    ("https://huggingface.co/datasets/mishig/sample_images/resolve/main/tiger.jpg", "a tiger"),
]

for url, caption in test_urls:
    try:
        img = Image.open(BytesIO(requests.get(url, timeout=10).content)).convert("RGB")
        working_data.append((img, caption))
        print(f"✓ {caption}")
    except:
        pass

# Process data
processed_data = []
for img, caption in working_data * 20:
    full_text = f"caption en {caption}"
    inputs = processor(text=full_text, images=img, return_tensors="pt", padding=True)
    labels = inputs["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100

    processed_data.append({
        "input_ids": inputs["input_ids"].squeeze(0),
        "attention_mask": inputs["attention_mask"].squeeze(0),
        "pixel_values": inputs["pixel_values"].squeeze(0),
        "labels": labels.squeeze(0)
    })

class PreprocessedDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

class PaddingCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id
    def __call__(self, features):
        return {
            "input_ids": pad_sequence([f["input_ids"] for f in features], batch_first=True, padding_value=self.pad_token_id),
            "attention_mask": pad_sequence([f["attention_mask"] for f in features], batch_first=True, padding_value=0),
            "labels": pad_sequence([f["labels"] for f in features], batch_first=True, padding_value=-100),
            "pixel_values": torch.stack([f["pixel_values"] for f in features])
        }

train_dataset = PreprocessedDataset(processed_data)

# Train
# Restart runtime, reload everything but change this:
training_args = TrainingArguments(
    output_dir="./paligemma-lora",
    num_train_epochs=1,  # Only 1 epoch
    per_device_train_batch_size=2,
    learning_rate=1e-4,  # Was 5e-4, now 5x lower
    logging_steps=5,
    bf16=True,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=PaddingCollator(processor.tokenizer.pad_token_id),
)

trainer.train()


`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


trainable params: 1,916,928 || all params: 2,925,383,408 || trainable%: 0.0655
✓ a red car
✓ beignets on a plate
✓ two parrots


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
Y

✓ a tree
✓ a tiger


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
Y

Step,Training Loss
5,21.809100
10,21.601900
15,21.265700
20,21.002900
25,20.403700
30,20.273600
35,20.128400
40,19.756400
45,19.606600
50,19.890200


TrainOutput(global_step=50, training_loss=20.573848419189453, metrics={'train_runtime': 161.5577, 'train_samples_per_second': 0.619, 'train_steps_per_second': 0.309, 'total_flos': 379575747338304.0, 'train_loss': 20.573848419189453, 'epoch': 1.0})

In [2]:
# Test the fine-tuned model
from PIL import Image
import requests
from io import BytesIO

test_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg"
image = Image.open(BytesIO(requests.get(test_url).content)).convert("RGB")

inputs = processor(text="caption en", images=image, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=20)
print(processor.decode(outputs[0], skip_special_tokens=True))


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


caption en
automobile model parked on a street


In [3]:
model.save_pretrained("./paligemma-finetuned-lora")
processor.save_pretrained("./paligemma-finetuned-lora")
print("✅ Model saved!")


✅ Model saved!


In [4]:
# Try the other images
test_url = "https://huggingface.co/datasets/mishig/sample_images/resolve/main/tiger.jpg"
image = Image.open(BytesIO(requests.get(test_url).content)).convert("RGB")
inputs = processor(text="caption en", images=image, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=20)
print(processor.decode(outputs[0], skip_special_tokens=True))


You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


caption en
animal lying down in the grass


In [5]:
!zip -r paligemma-finetuned-lora.zip paligemma-finetuned-lora/
from google.colab import files
files.download('paligemma-finetuned-lora.zip')

  adding: paligemma-finetuned-lora/ (stored 0%)
  adding: paligemma-finetuned-lora/preprocessor_config.json (deflated 48%)
  adding: paligemma-finetuned-lora/adapter_config.json (deflated 57%)
  adding: paligemma-finetuned-lora/README.md (deflated 66%)
  adding: paligemma-finetuned-lora/tokenizer.json (deflated 84%)
  adding: paligemma-finetuned-lora/tokenizer_config.json (deflated 96%)
  adding: paligemma-finetuned-lora/tokenizer.model (deflated 51%)
  adding: paligemma-finetuned-lora/added_tokens.json (stored 0%)
  adding: paligemma-finetuned-lora/adapter_model.safetensors (deflated 8%)
  adding: paligemma-finetuned-lora/special_tokens_map.json (deflated 78%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# Login to HuggingFace
from huggingface_hub import notebook_login
notebook_login()

# Push model
model.push_to_hub("Donald8585/paligemma-caption-finetuned")
processor.push_to_hub("Donald8585/paligemma-caption-finetuned")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   7%|7         |  556kB / 7.70MB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...p9lf18mpt/tokenizer.model:  98%|#########7| 4.17MB / 4.26MB            

  ...mp9lf18mpt/tokenizer.json:  73%|#######2  | 25.1MB / 34.6MB            

CommitInfo(commit_url='https://huggingface.co/Donald8585/paligemma-caption-finetuned/commit/0410e04d973c4d3409d2531f1b1d7505dea91c3e', commit_message='Upload processor', commit_description='', oid='0410e04d973c4d3409d2531f1b1d7505dea91c3e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Donald8585/paligemma-caption-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='Donald8585/paligemma-caption-finetuned'), pr_revision=None, pr_num=None)